In [5]:
pip install cvxpy

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 9.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/887.3 kB ? eta -:--:--
   ---------------------------------------- 887.3/887.3 kB 9.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/7.5 MB ? eta -:--:--
   ------------------- -------------------- 3.7/7.5 MB 18.2 MB/s eta 0:00:01
   ----------------------------- ---------- 5.5/7.5 MB 12.9 MB/s eta 0:00:01
   ---------------------------------------  7.3/7.5 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 7.5/7.5 MB 11.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# %% imports
import os
import numpy as np
import pandas as pd
import datetime as dt
from typing import List, Dict, Tuple

# data
import yfinance as yf

# TDA
from gtda.time_series import TakensEmbedding
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PersistenceLandscape

# Mapper
import kmapper as km
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Clustering
import hdbscan


In [10]:

# Optimization
import cvxpy as cp

# Metrics & utils
from sklearn.metrics import adjusted_rand_score
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

In [11]:
START_DATE = "2025-01-01"
END_DATE = "2025-10-08"
AN_DATE = "2025-07-08"

# URL de la lista del S&P 500
url = "https://www.slickcharts.com/sp500"


# Hacer la petición con headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
response = requests.get(url, headers=headers)
response.raise_for_status()  # Raise exception for bad status codes

# Parsear el HTML
soup = BeautifulSoup(response.text, 'html.parser')

# Encontrar la tabla
table = soup.find('table')

if table:
    # Leer la tabla con pandas
    df = pd.read_html(str(table))[0]
    
    # Extraer los tickers
    if 'Symbol' in df.columns:
        tickers = df['Symbol'].tolist()
        print(f"Found {len(tickers)} tickers")
        print("First 10 tickers:", tickers[:10])
    else:
        print("Available columns:", df.columns.tolist())

    # Extraer los pesos
    if 'Weight' in df.columns:
        weights = df['Weight'].tolist()
        print(f"Found {len(weights)} weights")
        print("First 10 weights:", weights[:10])
    else:
        print("Available columns:", df.columns.tolist())
else:
    print("No table found on the page")

# En la lista de ticker, reemplazar los "." por "-"
tickers = [ticker.replace('.', '-') for ticker in tickers]
print("Tickers after replacement:", tickers[:10])

Found 503 tickers
First 10 tickers: ['NVDA', 'MSFT', 'AAPL', 'AMZN', 'META', 'AVGO', 'GOOGL', 'GOOG', 'TSLA', 'BRK.B']
Found 503 weights
First 10 weights: ['7.86%', '6.47%', '6.41%', '3.93%', '3.03%', '2.83%', '2.69%', '2.52%', '2.46%', '1.67%']
Tickers after replacement: ['NVDA', 'MSFT', 'AAPL', 'AMZN', 'META', 'AVGO', 'GOOGL', 'GOOG', 'TSLA', 'BRK-B']


C:\Users\alfmi\AppData\Local\Temp\ipykernel_35780\3989306241.py:24: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]


In [14]:
"""
Mapper-based sparse portfolio selection pipeline

This script is a self-contained, ready-to-run Jupyter-style Python module that:
- downloads S&P500 price data (or loads a local CSV),
- computes Takens embeddings for each ticker,
- computes persistence summaries (max persistence) via giotto-tda,
- builds Mapper graphs (KeplerMapper) with PCA + max_persistence lenses,
- clusters points inside cover intervals with HDBSCAN,
- extracts exemplars / node representatives to form sparse portfolios,
- solves sparse Index-Tracking, MV and GMV on selected assets,
- evaluates out-of-sample performance with rolling windows (TE, Sharpe, turnover, ARI),
- includes configurable hyperparameters and utilities to run grid search.

Notes:
- This code assumes you will run it in an environment with internet access to fetch prices,
  or you can provide a local CSV of adjusted close prices.
- Main libraries: yfinance, pandas, numpy, scikit-learn, keplermapper, giotto-tda, hdbscan,
  cvxpy (or scipy.optimize) for QP, matplotlib.

Recommended: create a conda env and `pip install yfinance keplermapper giotto-tda hdbscan cvxpy scikit-learn matplotlib pandas numpy`

"""



# %% Configurable parameters (defaults and recommended ranges)
CONFIG = {
    # Data
    'start_date': '2010-01-01',
    'end_date': '2024-12-31',
    'data_source': 'yfinance',  # or 'local_csv'
    'local_csv_path': 'prices.csv',  # if using local CSV: rows=dates, cols=tickers, values=Adj Close

    # Rolling windows
    'in_sample_days': 126,   # ~6 months
    'out_of_sample_days': 21, # ~1 month
    'rebalance_step': 21,     # monthly

    # Takens
    'takens_dim': 4,
    'takens_delay': 1,

    # Persistence
    'homology_dimensions': [0, 1],  # compute H0 and H1

    # Mapper lenses
    'n_intervals': 20,
    'overlap': 0.35,
    'lens_components': 2,

    # clustering in Mapper cover
    'cluster_method': 'hdbscan',
    'hdbscan_min_cluster_size': 5,

    # selection rules
    'selection_per_node': 1,  # how many assets to pick per mapper-node
    'min_node_size': 3,       # ignore nodes smaller than this unless they include benchmark

    # optimization constraints
    'allow_short': False,
    'weight_bounds': (0.0, 1.0),

    # other
    'verbose': True,
}

# %% Utilities

def fetch_sp500_tickers() -> List[str]:
    """Fetch S&P500 tickers from Wikipedia. Requires internet."""
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    tables = pd.read_html(url)
    df = tables[0]
    tickers = df['Symbol'].tolist()
    # Some tickers contain dots (BRK.B) change to '-' for yfinance
    tickers = [t.replace('.', '-') for t in tickers]
    return tickers


def download_prices(tickers: List[str], start: str, end: str) -> pd.DataFrame:
    """Download adjusted close prices for given tickers using yfinance.
       Returns DataFrame indexed by date with columns = tickers.
    """
    data = yf.download(tickers, start=start, end=end, progress=False, threads=True)
    if ('Adj Close' in data.columns):
        prices = data['Adj Close']
    else:
        # single ticker case
        prices = data
    prices = prices.dropna(axis=1, how='all')
    return prices


# %% TDA functions

def compute_takens(series: np.ndarray, dim: int, tau: int) -> np.ndarray:
    """Given a 1D series (T,), return Takens embedding as (n_points, dim).
       Uses simple sliding window embedding.
    """
    T = len(series)
    m = dim
    t_max = T - (m - 1) * tau
    if t_max <= 0:
        raise ValueError('Time series too short for given embedding params')
    emb = np.zeros((t_max, m))
    for i in range(m):
        emb[:, i] = series[i * tau:i * tau + t_max]
    return emb

def persistence_max_persistence(point_cloud: np.ndarray, homology_dimensions=[0,1]) -> float:
    """Compute Vietoris-Rips persistence and return max persistence across dimensions."""
    vr = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
    diagrams = vr.fit_transform(point_cloud[None, ...])  # output shape varies by giotto-tda version
    # diagrams[0] is the diagrams for the first (and only) sample
    d0 = diagrams[0]
    max_p = 0.0
    # d0 can be: (n_dims, n_points, 2)  OR  (n_points, 2) when only one homology dimension returned
    if d0.ndim == 3:
        # iterate homology dimensions
        for dim in range(d0.shape[0]):
            d = d0[dim]
            if d.size == 0:
                continue
            lifetimes = d[:, 1] - d[:, 0]
            if lifetimes.size > 0:
                max_p = max(max_p, lifetimes.max())
    elif d0.ndim == 2:
        # single homology dimension
        if d0.size > 0:
            lifetimes = d0[:, 1] - d0[:, 0]
            if lifetimes.size > 0:
                max_p = float(lifetimes.max())
    return float(max_p)
# ...existing code...
def build_lenses(price_df: pd.DataFrame, tickers: List[str], config: Dict) -> pd.DataFrame:
    """Compute lenses for each ticker: lens1 = PCA1 of Takens embedding, lens2 = max_persistence."""
    te = TakensEmbedding(config['takens_dim'], config['takens_delay'])
    vr = VietorisRipsPersistence(homology_dimensions=config['homology_dimensions'])

    lens_rows = []
    for t in tickers:
        series = price_df[t].dropna().values
        if len(series) < config['takens_dim'] * config['takens_delay'] + 2:
            lens_rows.append({'ticker': t, 'pc1': np.nan, 'max_persistence': np.nan})
            continue
        s = (series - series.mean()) / (series.std() + 1e-9)
        emb = te.fit_transform(s.reshape(1, -1))[0]
        pca = PCA(n_components=1)
        pc1 = pca.fit_transform(emb).mean()
        # compute diagrams and robustly extract max persistence
        diagrams = vr.fit_transform(emb[None, ...])
        d0 = diagrams[0]
        max_p = 0.0
        if d0.ndim == 3:
            for dim in range(d0.shape[0]):
                d = d0[dim]
                if d.size == 0:
                    continue
                lifetimes = d[:, 1] - d[:, 0]
                if lifetimes.size > 0:
                    max_p = max(max_p, lifetimes.max())
        elif d0.ndim == 2:
            if d0.size > 0:
                lifetimes = d0[:, 1] - d0[:, 0]
                if lifetimes.size > 0:
                    max_p = float(lifetimes.max())
        lens_rows.append({'ticker': t, 'pc1': float(pc1), 'max_persistence': float(max_p)})

    lens_df = pd.DataFrame(lens_rows).set_index('ticker')
    lens_df = lens_df.apply(lambda x: (x - x.mean()) / (x.std() + 1e-9))
    return lens_df

def run_mapper(lens_df: pd.DataFrame, data_matrix: np.ndarray, config: Dict):
    """Construct Mapper graph using KeplerMapper. data_matrix is optional (e.g., embeddings or features per ticker).
       Returns KeplerMapper graph dict and the mapper object.
    """
    mapper = km.KeplerMapper(verbose=config['verbose'])
    lens = lens_df.values

    # use PCA of lenses if multiple components requested
    if lens.shape[1] > config['lens_components']:
        pca = PCA(n_components=config['lens_components'])
        lens_proj = pca.fit_transform(lens)
    else:
        lens_proj = lens

    cover = km.Cover(n_cubes=config['n_intervals'], perc_overlap=config['overlap'])

    if data_matrix is None:
        # KeplerMapper needs an array X; use lenses as proxy for neighborhood metric
        X_for_mapper = lens_proj
    else:
        X_for_mapper = data_matrix

    # clusterer: HDBSCAN
    clusterer = hdbscan.HDBSCAN(min_cluster_size=config['hdbscan_min_cluster_size'])

    graph = mapper.map(lens_proj, X_for_mapper, cover=cover, clusterer=clusterer)
    return mapper, graph


def extract_nodes(graph: Dict, min_node_size: int = 3) -> Dict[int, List[int]]:
    """Return nodes mapping: node_id -> list of sample indices (these indices correspond to order in lens_df X).
    """
    nodes = {}
    for node_id_str, sample_indices in graph['nodes'].items():
        # node_id_str is like 'cube_0-1_cluster_0' but keys are fine
        indices = graph['nodes'][node_id_str]
        if len(indices) >= min_node_size:
            nodes[node_id_str] = indices
    return nodes


# %% Selection rules (examples)

def select_exemplars_per_node(nodes: Dict[str, List[int]], tickers: List[str], lens_df: pd.DataFrame, selection_per_node: int = 1) -> List[str]:
    """Select exemplars per node by picking tickers closest to the node centroid in lens-space.
       nodes keys correspond to indices of lens_df rows (which preserve ordering of tickers list provided).
    """
    selected = []
    lens_vals = lens_df.values
    for node_id, indices in nodes.items():
        centroid = lens_vals[indices].mean(axis=0)
        dists = np.linalg.norm(lens_vals[indices] - centroid, axis=1)
        order = np.argsort(dists)
        picks = [tickers[indices[i]] for i in order[:selection_per_node]]
        selected.extend(picks)
    # unique
    return sorted(list(dict.fromkeys(selected)))


# %% Portfolio optimization helpers

def solve_index_tracking(prices: pd.DataFrame, tickers_subset: List[str], benchmark: pd.Series, allow_short=False) -> np.ndarray:
    """Quadratic program to minimize tracking error (variance of difference) with weights sum to 1 and bounds.
       Returns weights in same order as tickers_subset.
    """
    P = prices[tickers_subset].pct_change().dropna()
    r = P.mean().values  # not used for IT but kept
    # compute covariance of differences
    R_port = P.values  # T x n
    R_bench = benchmark.loc[P.index].values.reshape(-1, 1)  # T x 1
    D = R_port - R_bench
    Sigma_D = np.cov(D.T)
    n = len(tickers_subset)

    w = cp.Variable(n)
    obj = cp.quad_form(w, Sigma_D)
    constraints = [cp.sum(w) == 1]
    if not allow_short:
        constraints += [w >= 0]
    else:
        lb, ub = -10, 10
        constraints += [w >= lb, w <= ub]
    prob = cp.Problem(cp.Minimize(obj), constraints)
    prob.solve(solver=cp.OSQP, verbose=False)
    if w.value is None:
        # fallback equal weights
        return np.ones(n) / n
    return np.array(w.value).flatten()


def solve_gmv(prices: pd.DataFrame, tickers_subset: List[str], allow_short=False) -> np.ndarray:
    P = prices[tickers_subset].pct_change().dropna()
    Sigma = P.cov().values
    n = len(tickers_subset)
    w = cp.Variable(n)
    obj = cp.quad_form(w, Sigma)
    constraints = [cp.sum(w) == 1]
    if not allow_short:
        constraints += [w >= 0]
    prob = cp.Problem(cp.Minimize(obj), constraints)
    prob.solve(solver=cp.OSQP, verbose=False)
    if w.value is None:
        return np.ones(n) / n
    return np.array(w.value).flatten()


def solve_mv(prices: pd.DataFrame, tickers_subset: List[str], risk_aversion=1.0, allow_short=False) -> np.ndarray:
    P = prices[tickers_subset].pct_change().dropna()
    mu = P.mean().values
    Sigma = P.cov().values
    n = len(tickers_subset)
    w = cp.Variable(n)
    obj = -(mu @ w) + (risk_aversion / 2.0) * cp.quad_form(w, Sigma)
    constraints = [cp.sum(w) == 1]
    if not allow_short:
        constraints += [w >= 0]
    prob = cp.Problem(cp.Minimize(obj), constraints)
    prob.solve(solver=cp.OSQP, verbose=False)
    if w.value is None:
        return np.ones(n) / n
    return np.array(w.value).flatten()


# %% Evaluation metrics

def tracking_error(portfolio_returns: pd.Series, benchmark_returns: pd.Series) -> float:
    diff = portfolio_returns - benchmark_returns
    return float(np.sqrt(np.mean(diff ** 2)))


def sharpe(returns: pd.Series) -> float:
    mu = returns.mean()
    sigma = returns.std()
    if sigma == 0:
        return 0.0
    return float(mu / sigma * np.sqrt(252))


def turnover(weights_prev: np.ndarray, weights_new: np.ndarray) -> float:
    return float(np.sum(np.abs(weights_new - weights_prev)))


# %% Full backtest pipeline (rolling)

def backtest_mapper_pipeline(prices: pd.DataFrame, benchmark_ticker: str, config: Dict) -> Dict:
    """Main backtest loop: rolling in-sample -> build mapper -> select -> optimize -> evaluate out-of-sample.
       Returns dictionary with performance metrics time series.
    """
    dates = prices.index
    start = 0
    in_days = config['in_sample_days']
    out_days = config['out_of_sample_days']
    step = config['rebalance_step']

    results = []

    tickers = list(prices.columns)

    # initial previous weights (equal weight on universe)
    prev_weights = None

    for t0 in range(0, len(dates) - in_days - out_days + 1, step):
        in_slice = dates[t0:t0 + in_days]
        out_slice = dates[t0 + in_days:t0 + in_days + out_days]

        price_in = prices.loc[in_slice]
        price_out = prices.loc[out_slice]

        # build lenses for all tickers using in-sample window
        lens_df = build_lenses(price_in, tickers, config)
        # drop tickers with nan lens
        lens_df = lens_df.dropna()
        tickers_valid = list(lens_df.index)

        # create a data matrix for mapper: we use Takens embeddings averaged by feature per ticker
        # simpler: use lens values as X_for_mapper
        X_for_mapper = lens_df.values

        mapper, graph = run_mapper(lens_df, X_for_mapper, config)
        nodes = extract_nodes(graph, min_node_size=config['min_node_size'])

        # mapping from lens_df index order → tickers_valid
        # select exemplars
        selected = select_exemplars_per_node(nodes, tickers_valid, lens_df, selection_per_node=config['selection_per_node'])

        # ensure benchmark ticker included if present in selected nodes
        if benchmark_ticker in tickers_valid and benchmark_ticker not in selected:
            # find node containing benchmark and force include its exemplar
            for node_id, indices in nodes.items():
                if tickers_valid.index(benchmark_ticker) in indices:
                    picks = select_exemplars_per_node({node_id: indices}, tickers_valid, lens_df, selection_per_node=1)
                    selected.extend(picks)
                    break

        selected = sorted(list(set(selected)))
        if len(selected) == 0:
            # fallback: top 20 by market cap or equal weight sample of valid tickers
            selected = tickers_valid[:20]

        # Solve portfolios
        bench_ret_in = price_in[benchmark_ticker].pct_change().dropna()
        try:
            w_it = solve_index_tracking(price_in, selected, bench_ret_in, allow_short=config['allow_short'])
        except Exception as e:
            print('IT solver failed, fallback to equal weights', e)
            w_it = np.ones(len(selected)) / len(selected)

        try:
            w_gmv = solve_gmv(price_in, selected, allow_short=config['allow_short'])
        except Exception as e:
            print('GMV failed, fallback eq', e)
            w_gmv = np.ones(len(selected)) / len(selected)

        try:
            w_mv = solve_mv(price_in, selected, risk_aversion=1.0, allow_short=config['allow_short'])
        except Exception as e:
            print('MV failed, fallback eq', e)
            w_mv = np.ones(len(selected)) / len(selected)

        # compute out-of-sample returns for each portfolio
        P_out = price_out[selected].pct_change().dropna()
        bench_out = price_out[benchmark_ticker].pct_change().dropna()

        r_it = P_out.values @ w_it
        r_gmv = P_out.values @ w_gmv
        r_mv = P_out.values @ w_mv

        # aggregate metrics
        te = tracking_error(pd.Series(r_it, index=P_out.index), bench_out.loc[P_out.index])
        sr_it = sharpe(pd.Series(r_it, index=P_out.index))
        sr_gmv = sharpe(pd.Series(r_gmv, index=P_out.index))
        sr_mv = sharpe(pd.Series(r_mv, index=P_out.index))

        tr = turnover(prev_weights if prev_weights is not None and len(prev_weights)==len(w_it) else np.zeros_like(w_it), w_it)

        # ARI stability: build lens for next window and compare (simple approximation)
        # store results
        results.append({
            't0': in_slice[-1],
            'selected_count': len(selected),
            'selected': selected,
            'te_it': te,
            'sr_it': sr_it,
            'sr_gmv': sr_gmv,
            'sr_mv': sr_mv,
            'turnover_it': tr,
        })

        prev_weights = w_it

    return results


# %% Example run (main)
if __name__ == '__main__':
    # Config adjustments
    cfg = CONFIG.copy()
    cfg['start_date'] = '2012-01-01'
    cfg['end_date'] = '2024-06-30'
    cfg['n_intervals'] = 20
    cfg['overlap'] = 0.35
    cfg['takens_dim'] = 4
    cfg['takens_delay'] = 1
    cfg['hdbscan_min_cluster_size'] = 6
    cfg['in_sample_days'] = 126
    cfg['out_of_sample_days'] = 21
    cfg['rebalance_step'] = 21

    # Data fetch
    print('Fetching tickers...')
    
    print(f'{len(tickers)} tickers fetched')
    prices = download_prices(tickers, cfg['start_date'], cfg['end_date'])
    print('Prices downloaded, shape:', prices.shape)

    # choose a benchmark (SPY) or '^GSPC'
    benchmark = 'SPY'
    if benchmark not in prices.columns:
        # try to download benchmark
        prices_bench = yf.download(benchmark, start=cfg['start_date'], end=cfg['end_date'], progress=False)
        if 'Adj Close' in prices_bench.columns:
            prices[benchmark] = prices_bench['Adj Close']
        else:
            prices[benchmark] = prices_bench['Close']

    print('Running rolling backtest...')
    res = backtest_mapper_pipeline(prices, benchmark, cfg)
    print('Done. Number of rebalances:', len(res))

    # Simple summary
    df_res = pd.DataFrame(res)
    print(df_res[['t0', 'selected_count', 'te_it', 'sr_it', 'turnover_it']].head())
    df_res.to_csv('mapper_backtest_results.csv', index=False)
    print('Results saved to mapper_backtest_results.csv')


Fetching tickers...
503 tickers fetched


C:\Users\alfmi\AppData\Local\Temp\ipykernel_35780\2910462570.py:85: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start=start, end=end, progress=False, threads=True)


Prices downloaded, shape: (3142, 2515)


C:\Users\alfmi\AppData\Local\Temp\ipykernel_35780\2910462570.py:453: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices_bench = yf.download(benchmark, start=cfg['start_date'], end=cfg['end_date'], progress=False)


Running rolling backtest...
KeplerMapper(verbose=True)
Mapping on data shaped (2236, 2) using lens shaped (2236, 2)

Creating 400 hypercubes.

Created 238 edges and 157 nodes in 0:00:00.175623.
IT solver failed, fallback to equal weights ARPACK error -1: ARPACK error -1: No convergence (1531 iterations, 0/1 eigenvectors converged)


        CVXPY note: This failure was encountered while trying to certify
        that a matrix is positive semi-definite (see [1] for a definition).
        In rare cases, this method fails for numerical reasons even when the matrix is
        positive semi-definite. If you know that you're in that situation, you can
        replace the matrix A by cvxpy.psd_wrap(A).

        [1] https://en.wikipedia.org/wiki/Definite_matrix
        
GMV failed, fallback eq ARPACK error -1: ARPACK error -1: No convergence (1531 iterations, 0/1 eigenvectors converged)


        CVXPY note: This failure was encountered while trying to certify
        that a matrix is positive

...

IT solver failed, fallback to equal weights Quadratic form matrices must be symmetric/Hermitian.
GMV failed, fallback eq Quadratic form matrices must be symmetric/Hermitian.
MV failed, fallback eq Quadratic form matrices must be symmetric/Hermitian.
KeplerMapper(verbose=True)
Mapping on data shaped (2441, 2) using lens shaped (2441, 2)

Creating 400 hypercubes.

Created 239 edges and 164 nodes in 0:00:00.152962.
IT solver failed, fallback to equal weights ARPACK error -1: ARPACK error -1: No convergence (1571 iterations, 0/1 eigenvectors converged)


        CVXPY note: This failure was encountered while trying to certify
        that a matrix is positive semi-definite (see [1] for a definition).
        In rare cases, this method fails for numerical reasons even when the matrix is
        positive semi-definite. If you know that you're in that situation, you can
        replace the matrix A by cvxpy.psd_wrap(A).

        [1] https://en.wikipedia.org/wiki/Definite_matrix
        
GMV f

c:\Users\alfmi\anaconda3\envs\Python-9909\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


KeplerMapper(verbose=True)
Mapping on data shaped (2451, 2) using lens shaped (2451, 2)

Creating 400 hypercubes.

Created 325 edges and 195 nodes in 0:00:00.150480.
IT solver failed, fallback to equal weights ARPACK error -1: ARPACK error -1: No convergence (1801 iterations, 0/1 eigenvectors converged)


        CVXPY note: This failure was encountered while trying to certify
        that a matrix is positive semi-definite (see [1] for a definition).
        In rare cases, this method fails for numerical reasons even when the matrix is
        positive semi-definite. If you know that you're in that situation, you can
        replace the matrix A by cvxpy.psd_wrap(A).

        [1] https://en.wikipedia.org/wiki/Definite_matrix
        
GMV failed, fallback eq ARPACK error -1: ARPACK error -1: No convergence (1801 iterations, 0/1 eigenvectors converged)


        CVXPY note: This failure was encountered while trying to certify
        that a matrix is positive semi-definite (see [1] for 

c:\Users\alfmi\anaconda3\envs\Python-9909\Lib\site-packages\numpy\lib\function_base.py:2742: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]
c:\Users\alfmi\anaconda3\envs\Python-9909\Lib\site-packages\numpy\lib\function_base.py:2742: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]
c:\Users\alfmi\anaconda3\envs\Python-9909\Lib\site-packages\numpy\lib\function_base.py:2742: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]
c:\Users\alfmi\anaconda3\envs\Python-9909\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


IT solver failed, fallback to equal weights Quadratic form matrices must be symmetric/Hermitian.
GMV failed, fallback eq Quadratic form matrices must be symmetric/Hermitian.
MV failed, fallback eq Quadratic form matrices must be symmetric/Hermitian.
KeplerMapper(verbose=True)
Mapping on data shaped (2516, 2) using lens shaped (2516, 2)

Creating 400 hypercubes.

Created 297 edges and 186 nodes in 0:00:00.162914.
KeplerMapper(verbose=True)
Mapping on data shaped (2516, 2) using lens shaped (2516, 2)

Creating 400 hypercubes.

Created 337 edges and 201 nodes in 0:00:00.164829.
Done. Number of rebalances: 143
          t0  selected_count     te_it      sr_it  turnover_it
0 2012-07-02             153  0.029663   9.635049     1.000000
1 2012-08-01             171  0.000090   4.018496     1.000004
2 2012-08-30             172       inf        NaN     1.000000
3 2012-10-01             143  0.026853  10.584785     1.000000
4 2012-11-01             176  0.069934   5.105068     1.000000
Results 